# PubMed Oncologist — DPO Training

**Phase 2:** Direct Preference Optimization fine-tuning on top of the SFT LoRA.

DPO teaches the model WHAT NOT TO SAY by contrasting good and bad responses:
- Don't fabricate specific statistics or trial results
- Use general medical knowledge freely, but flag uncertainty when going beyond well-established evidence
- Self-correct when challenged
- Maintain clinical reasoning depth

**Philosophy:** The model should behave like a well-read oncologist — drawing on broad
medical training (general knowledge) while being transparent about the strength of evidence.
It should NOT behave like a closed-book QA system that refuses to engage unless a specific
source is cited. General, well-established medical knowledge is always fair game.

**Prerequisites:**
1. Run `pubmed_datagen_v2_jupyterlab.ipynb` fully (produces SFT + DPO data)
2. Train the SFT LoRA first (Phase 1)
3. GPU with 16+ GB VRAM (DGX Spark recommended)

**Pipeline:** Base Qwen3-14B → SFT LoRA (trained) → DPO LoRA (this notebook)

## 1. Setup — Configuration, Environment, GPU Check

Everything needed before training. Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [1]:
import os, sys, subprocess, importlib, importlib.util

# DGX Spark unified memory: force CUDA allocator reclamation instead of letting
# PyTorch treat all 128GB unified memory as endlessly cacheable CUDA memory.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.5,max_split_size_mb:256"

# ╔══════════════════════════════════════════════════════════════════════════════╗

# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║

# ║                                                                              ║

# ║  ORDER MATTERS:                                                              ║

# ║    1. torch check (no side effects)                                          ║

# ║    2. pip install small packages                                             ║

# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║

# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║

# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):

    """Run pip with given args, suppressing output unless it fails."""

    cmd = [sys.executable, "-m", "pip"] + list(args)

    env = os.environ.copy()

    if env_extra:

        env.update(env_extra)

    result = subprocess.run(cmd, capture_output=True, text=True, env=env)

    if result.returncode != 0:

        print(f"  PIP FAILED: {' '.join(args)}")

        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])

        return False

    return True

def _check_import(module_name):

    try:

        return importlib.import_module(module_name)

    except (ImportError, ModuleNotFoundError):

        return None

print("=" * 60)

print("ENVIRONMENT SETUP")

print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──────────────────────────────────────

import torch

if not torch.cuda.is_available():

    print("FATAL: torch.cuda.is_available() = False")

    print(f"  torch version: {torch.__version__}")

    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:

        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")

    else:

        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")

    raise RuntimeError("No GPU. Cannot continue. See messages above.")

_MEMORY_FRACTION = 0.55
try:
    torch.cuda.set_per_process_memory_fraction(_MEMORY_FRACTION, 0)
    _total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"CUDA memory cap: {_total_gb * _MEMORY_FRACTION:.1f} GB ({_MEMORY_FRACTION:.0%} of {_total_gb:.0f} GB) - allocator forced to reclaim")
except RuntimeError as exc:
    print(f"WARNING: could not set CUDA memory fraction: {exc}")
    print("Relying on PYTORCH_CUDA_ALLOC_CONF garbage collection only")
print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")

# ── 1b. Small utility packages ─────────────────────────────────────────────────

for module, install_args in {

    "psutil":      ["install", "-q", "psutil"],

    "matplotlib":  ["install", "-q", "matplotlib"],

    "ipywidgets":  ["install", "-q", "ipywidgets"],

    "torchvision": ["install", "-q", "--no-deps", "torchvision"],

    "PIL":         ["install", "-q", "pillow"],

}.items():

    if _check_import(module) is None:

        print(f"  Installing {install_args[-1]}...")

        _pip(*install_args)

# ── 1c. Fix causal_conv1d ──────────────────────────────────────────────────────

#   NGC ships causal_conv1d 1.6.0 Python pkg WITHOUT the compiled CUDA extension

#   (causal_conv1d_cuda). This causes a hard crash when transformers or unsloth

#   try to import FalconH1 model. We must fix this BEFORE importing either.

#

#   CRITICAL: pip caches a broken prebuilt aarch64 wheel. Must use --no-binary

#   to force a source build, plus CAUSAL_CONV1D_FORCE_BUILD=TRUE env var.

#   First build takes ~3 min on aarch64, then cached by pip for future runs.

_causal_ok = False

_build_env = {

    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",

    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",

}

try:

    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn

    _causal_ok = True

    print("  causal_conv1d: OK (CUDA extension loaded)")

    for _k in list(sys.modules.keys()):

        if "causal_conv1d" in _k:

            del sys.modules[_k]

except (ImportError, ModuleNotFoundError, OSError):

    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")

    _pip("uninstall", "-y", "causal-conv1d")

    _pip("cache", "remove", "causal_conv1d")

    for _k in list(sys.modules.keys()):

        if "causal_conv1d" in _k:

            del sys.modules[_k]

    importlib.invalidate_caches()

    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",

              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)

    if ok:

        importlib.invalidate_caches()

        try:

            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn

            _causal_ok = True

            print("  causal_conv1d: rebuilt OK (CUDA extension working)")

            for _k in list(sys.modules.keys()):

                if "causal_conv1d" in _k:

                    del sys.modules[_k]

        except (ImportError, ModuleNotFoundError, OSError):

            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")

            _pip("uninstall", "-y", "causal-conv1d")

            for _k in list(sys.modules.keys()):

                if "causal_conv1d" in _k:

                    del sys.modules[_k]

            importlib.invalidate_caches()

    else:

        print("  causal_conv1d: source build failed — uninstalling for fallback")

        _pip("uninstall", "-y", "causal-conv1d")

        importlib.invalidate_caches()

# ── 1d. Import unsloth FIRST, then transformers ────────────────────────────────

#   Unsloth MUST be imported before transformers/trl/peft to apply its

#   monkey-patches. Purge any pre-loaded transformers modules.

for _k in list(sys.modules.keys()):

    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):

        del sys.modules[_k]

importlib.invalidate_caches()

import unsloth

import transformers

print(f"  transformers {transformers.__version__}")

# ── 1e. Report final state ─────────────────────────────────────────────────────

print()

for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),

                   ("causal_conv1d", "causal_conv1d")]:

    m = _check_import(mod)

    v = getattr(m, "__version__", "installed") if m else "n/a"

    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"

    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗

# ║  STEP 2: CONFIGURATION                                                      ║

# ╚══════════════════════════════════════════════════════════════════════════════╝

print()

print("=" * 60)

print("CONFIGURATION")

print("=" * 60)

from pathlib import Path

# --- Paths (auto-detect Docker vs host) ---

if os.path.exists("/workspace/training/pubmed"):

    PROJECT_ROOT = Path("/workspace/training/pubmed")

    _env = "Docker (Unsloth container)"

elif os.path.exists("/workspace/pubmed"):

    PROJECT_ROOT = Path("/workspace/pubmed")

    _env = "Docker (legacy mount)"

else:

    PROJECT_ROOT = Path("/home/spark/projects/training/pubmed")

    _env = "Host (VS Code / venv)"

# =========================== MODEL CONFIGURATION ===========================

BASE_LLM        = "unsloth/Qwen3-14B-unsloth-bnb-4bit"

MODEL_NAME_BASE = "pubmed_oncologist_v2_dpo_qwen3_14b"

# =========================== INPUT DATA ===========================

DPO_DATA_FILE = PROJECT_ROOT / "data" / "training-data" / "pubmed_oncologist_v2_dpo.jsonl"

SFT_LORA_PATH = PROJECT_ROOT / "output" / "pubmed_oncologist_v2_sft" / "lora_adapters"

# =========================== OUTPUT DIRECTORIES ===========================

OUTPUT_BASE     = PROJECT_ROOT / "output" / MODEL_NAME_BASE

TRAIN_DIR       = OUTPUT_BASE / "train"

LORA_OUTPUT_DIR = OUTPUT_BASE / "lora_adapters"

# =========================== LoRA CONFIGURATION ===========================

LORA_R              = 32

LORA_ALPHA          = 32

LORA_DROPOUT        = 0

# Attention + MLP projections (matches Unsloth reference)

LORA_TARGET_MODULES = [

    "q_proj", "k_proj", "v_proj", "o_proj",

    "gate_proj", "up_proj", "down_proj",

]

# =========================== DPO TRAINING HYPERPARAMETERS ===========================

MAX_SEQ_LENGTH = 3072

BATCH_SIZE     = 1         # Smaller batch — DPO needs chosen+rejected per sample

GRAD_ACCUM     = 8         # Gradient accumulation (effective batch = 1×8 = 8)

LEARNING_RATE  = 5e-6      # Lower than SFT — fine adjustment, not major shift

DPO_BETA       = 0.05      # DPO temperature — lower = softer preference (allows broader knowledge use)

TARGET_EPOCHS  = 1

WARMUP_RATIO   = 0.1       # 10% warmup (DPO benefits from more warmup)

SAVE_STEPS     = 100

LOSS_TYPE      = "sigmoid"  # Standard DPO loss (alternatives: "hinge", "ipo")

# =========================== DATASET SAMPLING ===========================

# DPO is alignment, not knowledge — converges fast with fewer examples.

# Full dataset is ~31.7K pairs. Recommended sizes:

#   500-1000  = quick pipeline test

#   3000-5000 = proper test (evaluate alignment quality)

#   8000-12000 = production run

#   0         = use all pairs (not recommended — diminishing returns)

DPO_MAX_PAIRS  = 9000

# --- Print summary ---

print(f"  Environment:  {_env}")

print(f"  PROJECT_ROOT: {PROJECT_ROOT}")

print(f"  Base model:   {BASE_LLM}")

print(f"  Model name:   {MODEL_NAME_BASE}")

print(f"  SFT LoRA:     {SFT_LORA_PATH}")

print(f"  DPO data:     {DPO_DATA_FILE}")

print(f"  Output:       {LORA_OUTPUT_DIR}")

print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")

print(f"  DPO beta:     {DPO_BETA}")

print(f"  DPO max pairs: {DPO_MAX_PAIRS or 'ALL (no limit)'}")

print(f"  Effective batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")

print(f"  Learning rate: {LEARNING_RATE}")

print(f"  Max seq length: {MAX_SEQ_LENGTH}")

print(f"  Loss type:    {LOSS_TYPE}")

print(f"  Precision:    4-bit QLoRA (pre-quantized NF4)")

# --- Verify paths ---

for path, label in [(DPO_DATA_FILE, "DPO data"), (str(PROJECT_ROOT), "Project root")]:

    exists = os.path.exists(path)

    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")

    if not exists and label == "DPO data":

        raise FileNotFoundError(f"{label} not found: {path}\nRun pubmed_datagen_v2_jupyterlab.ipynb Section 10 first.")

if SFT_LORA_PATH.exists():

    print(f"  {'OK':7s} SFT LoRA: {SFT_LORA_PATH}")

else:

    print(f"  {'MISSING':7s} SFT LoRA: {SFT_LORA_PATH} (will train from base model)")

print()

print("Setup complete. All cells below can run sequentially.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.2.0

  unsloth                   2026.2.1             [OK]
  transformers              5.2.0                [OK]
  trl                       0.22.2               [OK]
  causal_conv1d             1.6.0                [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/pubmed
  Base model:   unsloth/Qwen3-14B-unsloth-bnb-4bit
  Model name:   pubmed_oncologist_v2_dpo_qwen3_14b
  SFT LoRA:     /workspace/training/pubmed/output/pubmed_oncologist_v2_sft/lora_adapters
  DPO data:     /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2_dpo.jsonl
  Output:       /workspace/training/pubmed/output/pubmed_oncologist_v2_dpo_qwen3_14b/lora_adapters
  LoRA:    

## 2. Load DPO Dataset

Load the DPO preference pairs produced by `pubmed_datagen_v2_jupyterlab.ipynb` (Section 10).
Format: `{chosen: [...messages], rejected: [...messages], source: str}`

In [2]:
import json
from datasets import Dataset as HFDataset

# Load the JSONL file
with open(DPO_DATA_FILE) as f:
    raw_pairs = [json.loads(line) for line in f]

print(f"Loaded {len(raw_pairs)} DPO pairs from {DPO_DATA_FILE.name}")

# Source distribution (before sampling)
from collections import Counter, defaultdict
sources = Counter(p.get('source', 'unknown') for p in raw_pairs)
print(f"\nSource distribution (full dataset):")
for src, cnt in sources.most_common():
    print(f"  {src:<25} {cnt:>5} ({cnt/len(raw_pairs)*100:.1f}%)")

# Stratified sampling — proportional by source, preserving distribution
if DPO_MAX_PAIRS and len(raw_pairs) > DPO_MAX_PAIRS:
    import random
    random.seed(3407)

    by_source = defaultdict(list)
    for p in raw_pairs:
        by_source[p.get('source', 'unknown')].append(p)

    total = len(raw_pairs)
    sampled = []
    for source, items in sorted(by_source.items()):
        n = max(1, round(DPO_MAX_PAIRS * len(items) / total))
        n = min(n, len(items))
        sampled.extend(random.sample(items, n))

    # Trim to exact target if rounding overshot
    if len(sampled) > DPO_MAX_PAIRS:
        random.shuffle(sampled)
        sampled = sampled[:DPO_MAX_PAIRS]

    print(f"\nStratified sampling: {total:,} → {len(sampled):,} pairs (DPO_MAX_PAIRS={DPO_MAX_PAIRS:,})")
    sampled_sources = Counter(p.get('source', 'unknown') for p in sampled)
    for src, cnt in sampled_sources.most_common():
        print(f"  {src:<25} {cnt:>5} ({cnt/len(sampled)*100:.1f}%)")

    raw_pairs = sampled
else:
    print(f"\nUsing all {len(raw_pairs):,} pairs (DPO_MAX_PAIRS={DPO_MAX_PAIRS or 'disabled'})")

Loaded 31651 DPO pairs from pubmed_oncologist_v2_dpo.jsonl

Source distribution (full dataset):
  beyond_evidence           14660 (46.3%)
  grounding_reject          14050 (44.4%)
  self_correction            2941 (9.3%)

Stratified sampling: 31,651 → 9,000 pairs (DPO_MAX_PAIRS=9,000)
  beyond_evidence            4169 (46.3%)
  grounding_reject           3995 (44.4%)
  self_correction             836 (9.3%)


## 3. Load Model & Tokenizer (4-bit)

Load Qwen3 14B in 4-bit. If SFT LoRA exists, load it as the starting point for DPO.

In [3]:
from unsloth import FastLanguageModel
import torch

print(f"Loading base model: {BASE_LLM}")

# Check if SFT LoRA exists
if SFT_LORA_PATH.exists():
    print(f"Loading SFT LoRA from: {SFT_LORA_PATH}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        str(SFT_LORA_PATH),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    print(f"✓ Loaded SFT LoRA adapter")
else:
    print(f"⚠ SFT LoRA not found at {SFT_LORA_PATH}")
    print(f"  Training DPO from base model (NOT RECOMMENDED — SFT first is better)")
    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_LLM,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )

# Fix pad token — set pad = eos for causal LM training.
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")

# Align model config so trainer doesn't warn about token mismatch
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

print(f"\n✓ Model loaded")
print(f"  Precision: 4-bit QLoRA (pre-quantized NF4)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading base model: unsloth/Qwen3-14B-unsloth-bnb-4bit
Loading SFT LoRA from: /workspace/training/pubmed/output/pubmed_oncologist_v2_sft/lora_adapters
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

Unsloth 2026.2.1 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


✓ Loaded SFT LoRA adapter
  Set pad_token = eos_token ('<|im_end|>', id=151645)

✓ Model loaded
  Precision: 4-bit QLoRA (pre-quantized NF4)
  Max sequence length: 3072
  Vocab size: 151669
  GPU allocated: 11.7 GB


## 4. Validate & Prepare DPO Dataset

Convert chat-message format to the prompt/chosen/rejected text format
expected by TRL's DPOTrainer. Uses the tokenizer loaded in step 3.

In [4]:
# ============================================================================
# VALIDATE AND CONVERT DPO PAIRS
# ============================================================================
# TRL DPOTrainer expects: {prompt: str, chosen: str, rejected: str}
# Our data has: {chosen: [messages], rejected: [messages]}
# We apply the chat template to convert messages → formatted text.
# Uses the tokenizer already loaded from the model cell above.
#
# Pairs exceeding MAX_SEQ_LENGTH tokens are filtered out so no truncation
# happens during training.
# ============================================================================

errors = []
formatted_pairs = []
skipped_too_long = 0

for i, pair in enumerate(raw_pairs):
    chosen_msgs = pair.get("chosen", [])
    rejected_msgs = pair.get("rejected", [])

    # Validate structure
    if len(chosen_msgs) < 3 or len(rejected_msgs) < 3:
        errors.append(f"Pair {i}: too few messages (chosen={len(chosen_msgs)}, rejected={len(rejected_msgs)})")
        continue

    # Verify chosen and rejected share the same prompt
    if chosen_msgs[:-1] != rejected_msgs[:-1]:
        chosen_prompt = chosen_msgs[:-1]
        rejected_prompt = rejected_msgs[:-1]
        if len(chosen_prompt) != len(rejected_prompt):
            errors.append(f"Pair {i}: prompt length mismatch")
            continue
        else:
            errors.append(f"Pair {i}: prompt content mismatch (same length, different messages)")
            continue

    # Extract prompt (system + user) and responses
    prompt_msgs = chosen_msgs[:-1]
    chosen_response = chosen_msgs[-1]["content"]
    rejected_response = rejected_msgs[-1]["content"]

    # Format prompt using chat template
    # enable_thinking=False because <think> is already in the response text.
    # Without this, the template adds <think> to the prompt AND the response
    # starts with <think> → double <think> tag → corrupted training data.
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # Filter out pairs that exceed MAX_SEQ_LENGTH (prompt + longer response)
    p_len = len(tokenizer.encode(prompt_text, add_special_tokens=False))
    c_len = len(tokenizer.encode(chosen_response, add_special_tokens=False))
    r_len = len(tokenizer.encode(rejected_response, add_special_tokens=False))
    if p_len + max(c_len, r_len) > MAX_SEQ_LENGTH:
        skipped_too_long += 1
        continue

    formatted_pairs.append({
        "prompt": prompt_text,
        "chosen": chosen_response,
        "rejected": rejected_response,
    })

# Report
if errors:
    print(f"⚠ {len(errors)} validation errors:")
    for e in errors[:5]:
        print(f"  {e}")
    if len(errors) > 5:
        print(f"  ... and {len(errors) - 5} more")
else:
    print(f"✓ All {len(raw_pairs)} pairs passed validation")

if skipped_too_long:
    print(f"⚠ Filtered {skipped_too_long} pairs exceeding MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")

print(f"\nFormatted DPO pairs: {len(formatted_pairs)}")

# Convert to HuggingFace Dataset
dpo_dataset = HFDataset.from_list(formatted_pairs)
print(f"Dataset: {dpo_dataset}")
print(f"Columns: {dpo_dataset.column_names}")

# Length stats
chosen_lens = [len(p['chosen']) for p in formatted_pairs]
rejected_lens = [len(p['rejected']) for p in formatted_pairs]
prompt_lens = [len(p['prompt']) for p in formatted_pairs]

print(f"\nLength stats (chars):")
print(f"  Prompt:   avg={sum(prompt_lens)//len(prompt_lens):,}, max={max(prompt_lens):,}")
print(f"  Chosen:   avg={sum(chosen_lens)//len(chosen_lens):,}, max={max(chosen_lens):,}")
print(f"  Rejected: avg={sum(rejected_lens)//len(rejected_lens):,}, max={max(rejected_lens):,}")

# Sample
print(f"\nSample (first pair):")
print(f"  Prompt: {formatted_pairs[0]['prompt'][:200]}...")
print(f"  Chosen: {formatted_pairs[0]['chosen'][:200]}...")
print(f"  Rejected: {formatted_pairs[0]['rejected'][:200]}...")

del raw_pairs

✓ All 9000 pairs passed validation
⚠ Filtered 3077 pairs exceeding MAX_SEQ_LENGTH=3072

Formatted DPO pairs: 5923
Dataset: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 5923
})
Columns: ['prompt', 'chosen', 'rejected']

Length stats (chars):
  Prompt:   avg=2,040, max=2,178
  Chosen:   avg=9,935, max=13,553
  Rejected: avg=5,248, max=13,073

Sample (first pair):
  Prompt: <|im_start|>system
You are a clinical oncologist with deep expertise in cancer biology, diagnosis, treatment planning, and patient care. You have extensive experience across medical oncology, surgical...
  Chosen: <think>
Okay, the user is asking about AxonFinder's accuracy compared to other methods for segmenting axons in prostate cancer tissues. Let me start by understanding the context. The reference abstrac...
  Rejected: AxonFinder stands out as a superior method for segmenting axons in prostate cancer tissues compared to currently available techniques. Our comparative analysis, which in

## 5. Prepare SFT LoRA for DPO Training

The model already has SFT LoRA adapters loaded. Instead of creating new adapters
(which would stack on top), we continue training the **same** SFT LoRA weights
with DPO. This produces a **single LoRA adapter** relative to base Qwen3-14B
that contains both SFT and DPO training — exactly what vLLM needs.

In [5]:
# No get_peft_model() — we keep the existing SFT LoRA adapters
# and continue training them with DPO. This produces a single LoRA
# relative to base Qwen3-14B (SFT + DPO combined).
FastLanguageModel.for_training(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = trainable / total * 100

print(f"✓ SFT LoRA adapters ready for DPO training (no new adapters created)")
print(f"✓ Trainable: {trainable:,} / {total:,} params ({pct:.2f}%)")
print(f"✓ Target modules: {LORA_TARGET_MODULES}")

✓ SFT LoRA adapters ready for DPO training (no new adapters created)
✓ Trainable: 128,450,560 / 8,691,809,280 params (1.48%)
✓ Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 6. DPO Trainer Setup

Configure TRL's DPOTrainer. Key differences from SFT:
- **beta**: Controls how strongly the model should prefer chosen over rejected
- **Learning rate**: Much lower than SFT (5e-6 vs 1e-4)
- **No reference model**: Unsloth handles implicit reference via adapter isolation

In [6]:
from trl import DPOTrainer, DPOConfigimport math# Calculate training stepseffective_batch = BATCH_SIZE * GRAD_ACCUMsteps_per_epoch = math.ceil(len(dpo_dataset) / effective_batch)max_steps = steps_per_epoch * TARGET_EPOCHSwarmup_steps = int(max_steps * WARMUP_RATIO)print(f"DPO Training Plan")print(f"  DPO pairs:          {len(dpo_dataset)}")print(f"  Effective batch:    {effective_batch}")print(f"  Steps per epoch:    {steps_per_epoch}")print(f"  Total steps:        {max_steps}")print(f"  Warmup steps:       {warmup_steps}")print(f"  DPO beta:           {DPO_BETA}")# Create output directoriesTRAIN_DIR.mkdir(parents=True, exist_ok=True)# Fix: TRL DPOTrainer expects model.warnings_issued dict, but PEFT/Unsloth# wrapper does not expose it. Must bypass __setattr__ with object.__setattr__.if not hasattr(model, "warnings_issued"):    object.__setattr__(model, "warnings_issued", {})    if hasattr(model, "base_model"):        object.__setattr__(model.base_model, "warnings_issued", {})        if hasattr(model.base_model, "model"):            model.base_model.model.warnings_issued = {}trainer = DPOTrainer(    model=model,    ref_model=None,       # Unsloth handles reference model internally    args=DPOConfig(        # DPO-specific        beta=DPO_BETA,        loss_type=LOSS_TYPE,        max_length=MAX_SEQ_LENGTH,        max_prompt_length=MAX_SEQ_LENGTH // 2,precompute_ref_log_probs=True,precompute_ref_batch_size=1,        # Batching        per_device_train_batch_size=BATCH_SIZE,        gradient_accumulation_steps=GRAD_ACCUM,        # Learning rate        learning_rate=LEARNING_RATE,        lr_scheduler_type="cosine",        warmup_steps=warmup_steps,        # Steps        max_steps=max_steps,        # Precision        fp16=not torch.cuda.is_bf16_supported(),        bf16=torch.cuda.is_bf16_supported(),        # Optimizer        optim="adamw_8bit",        weight_decay=0.01,        seed=3407,        gradient_checkpointing=True,dataloader_pin_memory=False,        # Checkpointing        output_dir=str(TRAIN_DIR),        save_strategy="steps",        save_steps=SAVE_STEPS,        save_total_limit=3,        # Logging        logging_steps=5,        report_to="none",        dataset_num_proc=1,    ),    train_dataset=dpo_dataset,    tokenizer=tokenizer,)print(f"\n✓ DPO Trainer configured")print("  precompute_ref_log_probs: True")print("  precompute_ref_batch_size: 1")print("  dataloader_pin_memory: False")print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

DPO Training Plan
  DPO pairs:          5923
  Effective batch:    8
  Steps per epoch:    741
  Total steps:        741
  Warmup steps:       74
  DPO beta:           0.05


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/5923 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/5923 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/5923 [00:00<?, ? examples/s]


✓ DPO Trainer configured
  Precision: bf16


## 7. Train!

DPO training. Watch the loss — it typically starts around 0.65-0.70 (log 2)
and decreases to 0.40-0.55.

**Key metrics to watch:**
- `train/loss`: Should decrease steadily
- `train/rewards/chosen`: Should increase (model prefers chosen)
- `train/rewards/rejected`: Should decrease (model avoids rejected)
- `train/rewards/margins`: Chosen - Rejected gap (should widen)

**Expected time:** 30-60 min on DGX Spark (depends on dataset size)

In [7]:
from transformers.trainer_utils import get_last_checkpoint

print("DPO Training started...")
print(f"Watch for loss to decrease from ~0.69 toward ~0.40-0.55")
print(f"Reward margins should widen (chosen > rejected)")
print()

last_ckpt = get_last_checkpoint(trainer.args.output_dir)
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=True)
else:
    print("No previous checkpoint found — starting fresh.")
    result = trainer.train()

print(f"\n✓ DPO Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Total steps:    {result.global_step}")
print(f"  Training time:  {result.metrics.get('train_runtime', 0) / 60:.1f} minutes")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


DPO Training started...
Watch for loss to decrease from ~0.69 toward ~0.40-0.55
Reward margins should widen (chosen > rejected)

No previous checkpoint found — starting fresh.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,923 | Num Epochs = 1 | Total steps = 741
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 128,450,560 of 14,896,757,760 (0.86% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
5,4.581554,60.174866,25.453064,0.725000,34.721806,-1917.969971,-764.075378,-2.138325,-1.864864,0,0,0
10,9.847673,60.810139,39.177517,0.550000,21.632626,-1881.071533,-983.313660,-2.198800,-1.983310,No Log,No Log,No Log
15,6.210699,57.811729,28.154705,0.675000,29.657024,-1829.712158,-755.066772,-2.189519,-1.863738,No Log,No Log,No Log
20,3.950831,61.047253,26.743322,0.725000,34.303932,-1839.583252,-761.731201,-2.190268,-1.879762,No Log,No Log,No Log
25,3.264677,58.497215,22.884949,0.775000,35.612270,-1920.813843,-739.891724,-2.152945,-1.799389,No Log,No Log,No Log
30,6.453645,63.594982,47.853863,0.450000,15.741114,-1829.924194,-1085.455322,-2.210682,-2.075494,No Log,No Log,No Log
35,7.732491,62.438240,43.730690,0.475000,18.707550,-1805.366577,-1023.980774,-2.188900,-2.038264,No Log,No Log,No Log
40,4.393958,62.516590,31.098846,0.650000,31.417734,-1845.863647,-832.862488,-2.185559,-1.889890,No Log,No Log,No Log
45,3.933853,64.319443,31.411488,0.725000,32.907959,-1821.313110,-822.069031,-2.206929,-1.950909,No Log,No Log,No Log
50,3.771233,63.617798,34.587433,0.625000,29.030365,-1871.210938,-894.748535,-2.182307,-1.914058,No Log,No Log,No Log



✓ DPO Training complete!
  Final loss:     0.7609
  Total steps:    741
  Training time:  1071.7 minutes


## 8. Save DPO LoRA Adapter

Save the combined SFT+DPO LoRA adapter. This is a **single LoRA** relative to
the base Qwen3-14B model — load it directly in vLLM with no stacking or merging required.

In [8]:
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

print(f"DPO LoRA adapter saved to: {LORA_OUTPUT_DIR}")
print()
total_size = 0
for f in sorted(LORA_OUTPUT_DIR.iterdir()):
    size = f.stat().st_size
    total_size += size
    print(f"  {f.name:<40s} {size:>12,} bytes")
print(f"  {'─' * 52}")
print(f"  {'TOTAL':<40s} {total_size:>12,} bytes ({total_size / 1024 / 1024:.1f} MB)")

print(f"\n✓ DPO training pipeline complete!")
print(f"\nDeployment (single LoRA on base model):")
print(f"  Base model:  {BASE_LLM}")
print(f"  LoRA:        {LORA_OUTPUT_DIR}")
print(f"\nvLLM serving:")
print(f"  vllm serve Qwen/Qwen3-14B \\")
print(f"    --enable-lora \\")
print(f"    --lora-modules oncologist={LORA_OUTPUT_DIR}")

DPO LoRA adapter saved to: /workspace/training/pubmed/output/pubmed_oncologist_v2_dpo_qwen3_14b/lora_adapters

  README.md                                       5,244 bytes
  adapter_config.json                             1,199 bytes
  adapter_model.safetensors                 513,877,864 bytes
  chat_template.jinja                             4,673 bytes
  tokenizer.json                             11,422,924 bytes
  tokenizer_config.json                             371 bytes
  ────────────────────────────────────────────────────
  TOTAL                                     525,312,275 bytes (501.0 MB)

✓ DPO training pipeline complete!

Deployment (single LoRA on base model):
  Base model:  unsloth/Qwen3-14B-unsloth-bnb-4bit
  LoRA:        /workspace/training/pubmed/output/pubmed_oncologist_v2_dpo_qwen3_14b/lora_adapters

vLLM serving:
  vllm serve Qwen/Qwen3-14B \
    --enable-lora \
    --lora-modules oncologist=/workspace/training/pubmed/output/pubmed_oncologist_v2_dpo_qwen3_14b/l

## 9. Quick Evaluation

Test the DPO-trained model on the key behaviors it should have learned:
1. Clinical reasoning that draws on broad medical knowledge (not just cited sources)
2. Transparency about evidence strength (distinguishes well-established facts from emerging data)
3. Self-correction when challenged

In [9]:
# ============================================================================
# QUICK EVALUATION — test DPO alignment on key behaviors
# ============================================================================
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Test prompts targeting DPO-trained behaviors
EVAL_PROMPTS = [
    # Test 1: Should use medical knowledge freely while noting evidence quality
    {
        "name": "Knowledge + evidence awareness",
        "system": "You are a clinical oncologist specializing in breast cancer.",
        "user": "What is the 5-year survival rate for stage III triple-negative breast cancer treated with pembrolizumab plus chemotherapy based on the KEYNOTE-522 trial?",
        "check": "Should draw on medical knowledge to answer, noting evidence source/strength where appropriate",
    },
    # Test 2: Should engage with the question using general knowledge, not refuse
    {
        "name": "Broad reasoning (not source-restricted)",
        "system": "You are a clinical oncologist.",
        "user": "Based on a single case report of a patient with KRAS G12C mutated pancreatic cancer, what is the expected response rate to sotorasib monotherapy across all pancreatic cancer patients?",
        "check": "Should engage using broader KRAS inhibitor knowledge, while noting the case report's limitations — NOT refuse to answer just because only one source is cited",
    },
    # Test 3: Should self-correct when challenged
    {
        "name": "Self-correction",
        "system": "You are a clinical oncologist.",
        "user": "You previously said tamoxifen works by blocking HER2 receptors. That doesn't sound right — can you reconsider?",
        "check": "Should correct the error (tamoxifen blocks estrogen receptors, not HER2)",
    },
]

print("DPO BEHAVIORAL EVALUATION")
print("=" * 60)

for test in EVAL_PROMPTS:
    messages = [
        {"role": "system", "content": test["system"]},
        {"role": "user", "content": test["user"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"\n{'='*60}")
    print(f"  TEST: {test['name'].upper()}")
    print(f"  [CHECK] {test['check']}")
    print(f"  [USER] {test['user'][:150]}")
    print(f"  [MODEL] ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

print("=" * 60)
print("Review the responses above to verify DPO alignment.")
print("Pay attention to: broad medical reasoning, evidence transparency, error correction.")

DPO BEHAVIORAL EVALUATION

  TEST: KNOWLEDGE + EVIDENCE AWARENESS
  [CHECK] Should draw on medical knowledge to answer, noting evidence source/strength where appropriate
  [USER] What is the 5-year survival rate for stage III triple-negative breast cancer treated with pembrolizumab plus chemotherapy based on the KEYNOTE-522 tri
  [MODEL] 

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 1075, in launch

<think>
Okay, the user is asking about the 5-year survival rate for stage III triple-negative breast cancer (TNBC) treated with pembrolizumab plus chemotherapy based on the KEYNOTE-522 trial. Let me start by recalling what I know about this trial.

First, KEYNOTE-522 was a phase III study comparing pembrolizumab plus chemo versus chemo alone in early-stage TNBC. The initial results were presented at ASCO 2020, but the 5-year follow-up is crucial here since the user specifically asked for the 5-year survival rate. I need to make sure I have the latest data from that trial.

I remember that the primary endpoints were pathologic complete response (pCR) and event-free survival (EFS). The 5-year EFS was reported in later publications. From what I recall, the 5-year EFS was around 64.8% in the pembrolizumab arm versus 51.2% in the placebo arm. But I should verify the exact numbers because precision is key here.

Also, the user mentioned stage III TNBC. Stage III in breast cancer typically in